## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE


## 2. Load Dataset

In [2]:
train_df=pd.read_csv("../data/raw/aps_failure_training_set.csv",skiprows=20,na_values="na")

In [3]:
df_preprocessed = train_df.copy()

## 3. Basic Dataset Information

In [4]:
df_preprocessed.shape

(60000, 171)

In [5]:
df_preprocessed.head()

,class,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,76698,NaN,2.130706e+09,280.0,0.0,0.0,0.0,0.0,0.0,...,1240520.0,493384.0,721044.0,469792.0,339156.0,157956.0,73224.0,0.0,0.0,0.0
1,neg,33058,NaN,0.000000e+00,NaN,0.0,0.0,0.0,0.0,0.0,...,421400.0,178064.0,293306.0,245416.0,133654.0,81140.0,97576.0,1500.0,0.0,0.0
2,neg,41040,NaN,2.280000e+02,100.0,0.0,0.0,0.0,0.0,0.0,...,277378.0,159812.0,423992.0,409564.0,320746.0,158022.0,95128.0,514.0,0.0,0.0
3,neg,12,0.0,7.000000e+01,66.0,0.0,10.0,0.0,0.0,0.0,...,240.0,46.0,58.0,44.0,10.0,0.0,0.0,0.0,4.0,32.0
4,neg,60874,NaN,1.368000e+03,458.0,0.0,0.0,0.0,0.0,0.0,...,622012.0,229790.0,405298.0,347188.0,286954.0,311560.0,433954.0,1218.0,0.0,0.0


In [6]:
df_preprocessed.info()

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 171 entries, class to eg_000
dtypes: float64(169), int64(1), str(1)
memory usage: 78.3 MB


## 4. Check Missing Values

In [7]:
df_preprocessed.isnull().sum()

class         0
aa_000        0
ab_000    46329
ac_000     3335
ad_000    14861
          ...  
ee_007      671
ee_008      671
ee_009      671
ef_000     2724
eg_000     2723
Length: 171, dtype: int64

In [8]:
df_preprocessed.isnull().sum().sort_values(ascending=False).head(20)

br_000    49264
bq_000    48722
bp_000    47740
bo_000    46333
ab_000    46329
cr_000    46329
bn_000    44009
bm_000    39549
bl_000    27277
bk_000    23034
ad_000    14861
cg_000    14861
ch_000    14861
cf_000    14861
co_000    14861
cx_000    13808
cz_000    13808
cy_000    13808
dc_000    13808
db_000    13808
dtype: int64

## Check Duplicated Rows

In [9]:
df_preprocessed.duplicated().sum()

np.int64(0)

## Separate Features and Target

In [83]:
x=df_preprocessed.drop(columns=["class"])
y=df_preprocessed["class"]

In [84]:
y.value_counts()

class
neg    59000
pos     1000
Name: count, dtype: int64

## Label Encoding


The target contains two classes:

- `neg` → No APS failure
- `pos` → APS failure

We use LabelEncoder to convert them into numerical values.

In [85]:
le=LabelEncoder()
y=le.fit_transform(y)

In [86]:
le.classes_

"""
neg → 0
pos → 1
"""

'\nneg → 0\npos → 1\n'

## Train-Test-Split

In [87]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [88]:
x_train.head()

,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,ag_003,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
39839,30606,NaN,3.260000e+02,226.0,0.0,0.0,0.0,0.0,0.0,0.0,...,290114.0,120320.0,246698.0,303534.0,228414.0,97398.0,60828.0,0.0,0.0,0.0
57649,314,0.0,1.400000e+01,14.0,0.0,0.0,0.0,0.0,0.0,2738.0,...,3222.0,1350.0,1632.0,1692.0,3288.0,2254.0,2.0,0.0,0.0,0.0
11862,3572,NaN,0.000000e+00,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,13308.0,7196.0,25950.0,117070.0,662.0,0.0,0.0,0.0,0.0,0.0
9381,33020,NaN,2.130706e+09,88.0,0.0,0.0,0.0,0.0,0.0,0.0,...,371136.0,185272.0,385210.0,329970.0,196806.0,82266.0,2992.0,0.0,0.0,0.0
43612,2518,NaN,5.000000e+01,34.0,0.0,0.0,0.0,0.0,1766.0,29390.0,...,9592.0,4752.0,13872.0,23130.0,29126.0,28378.0,88.0,0.0,0.0,0.0


In [89]:
x_train.isnull().sum()

aa_000        0
ab_000    37031
ac_000     2661
ad_000    11870
ae_000     1986
          ...  
ee_007      525
ee_008      525
ee_009      525
ef_000     2170
eg_000     2170
Length: 170, dtype: int64

In [90]:
x_train.isnull().sum().sum()

np.int64(679122)

In [91]:
y_train

array([0, 0, 0, ..., 0, 0, 0], shape=(48000,))

In [92]:
print("X_train:", x_train.shape)
print("X_test:", x_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (48000, 170)
X_test: (12000, 170)
y_train: (48000,)
y_test: (12000,)


## Handle Missing Values-->SimpleImputer

In [93]:

si=SimpleImputer(strategy="mean")

x_train_imputed=si.fit_transform(x_train)
x_test_imputed=si.transform(x_test)

In [94]:
x_train_imputed=pd.DataFrame(x_train_imputed,columns=x_train.columns,index=x_train.index)

In [97]:
x_test_imputed=pd.DataFrame(x_test_imputed,columns=x_test.columns,index=x_test.index)

In [95]:
x_train_imputed.head()

,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,ag_003,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
39839,30606.0,0.710183,3.260000e+02,226.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,290114.0,120320.0,246698.0,303534.0,228414.0,97398.0,60828.0,0.0,0.0,0.0
57649,314.0,0.000000,1.400000e+01,14.000000,0.0,0.0,0.0,0.0,0.0,2738.0,...,3222.0,1350.0,1632.0,1692.0,3288.0,2254.0,2.0,0.0,0.0,0.0
11862,3572.0,0.710183,0.000000e+00,238046.204816,0.0,0.0,0.0,0.0,0.0,0.0,...,13308.0,7196.0,25950.0,117070.0,662.0,0.0,0.0,0.0,0.0,0.0
9381,33020.0,0.710183,2.130706e+09,88.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,371136.0,185272.0,385210.0,329970.0,196806.0,82266.0,2992.0,0.0,0.0,0.0
43612,2518.0,0.710183,5.000000e+01,34.000000,0.0,0.0,0.0,0.0,1766.0,29390.0,...,9592.0,4752.0,13872.0,23130.0,29126.0,28378.0,88.0,0.0,0.0,0.0


In [98]:
x_test_imputed.head()

,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,ag_003,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
51064,40966.0,0.710183,1.900000e+02,130.000000,0.0000,0.000000,0.0,0.0,0.0,0.0,...,471848.0,250870.0,528518.0,499144.0,147948.0,23158.0,24128.0,3332.0,0.000000,0.000000
42770,32.0,0.710183,0.000000e+00,238046.204816,0.0000,0.000000,0.0,0.0,0.0,0.0,...,650.0,62.0,90.0,40.0,0.0,0.0,0.0,0.0,0.000000,0.000000
54787,868.0,2.000000,6.400000e+01,40.000000,0.0000,0.000000,0.0,0.0,0.0,0.0,...,6176.0,2026.0,4678.0,3384.0,3308.0,17114.0,120.0,0.0,0.000000,0.000000
22821,52160.0,0.710183,6.380000e+02,552.000000,0.0000,0.000000,0.0,0.0,0.0,0.0,...,535892.0,268874.0,518830.0,477658.0,433820.0,154602.0,28362.0,1122.0,0.000000,0.000000
29074,419854.0,0.710183,3.581963e+08,238046.204816,7.2135,11.365454,0.0,0.0,0.0,0.0,...,3780230.0,1825496.0,3818800.0,3276700.0,2489198.0,1539562.0,1858406.0,52684.0,0.096705,0.223129


In [99]:
x_train_imputed.isnull().sum()

aa_000    0
ab_000    0
ac_000    0
ad_000    0
ae_000    0
         ..
ee_007    0
ee_008    0
ee_009    0
ef_000    0
eg_000    0
Length: 170, dtype: int64

In [100]:
x_train_imputed.isnull().sum().sum()

np.int64(0)

In [101]:
x_test_imputed.isnull().sum().sum()

np.int64(0)